# RNN Training + ONNX Export for MetaTrader 5

**Что делает ноутбук:**
1. Загружает CSV (data_BUY/SELL) из Google Drive
2. Тренирует GRU-сети с фиксированными гиперпараметрами
3. Бэктест: WinRate, MaxDD, Profit Factor, Z-Score
4. Score: **PF × WR × (1−DD) × √(trades / (months×3))**
5. Экспортирует модели в ONNX + RNN_Scaler.mqh

**Структура Drive:**
```
Мой Диск / othercomputers/Ноутбук/RNN /
  data/          ← кинуть CSV сюда
  output/        ← результат
```

## Подключение Google Drive и базовые импорты

## 0. Проверка и установка зависимостей

Запускайте эту ячейку первой после каждого нового или перезапущенного Colab runtime. Она проверит окружение, установит отсутствующие пакеты и выведет версии.

In [1]:
# @title 0. Check / Install dependencies
import importlib.util
import subprocess
import sys

required = {
    'onnx': 'onnx',
    'onnxruntime': 'onnxruntime',
    'onnxscript': 'onnxscript',
}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print('Устанавливаю отсутствующие пакеты:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('Все необходимые пакеты уже установлены.')

import onnx
import onnxruntime
import onnxscript

print(f'onnx:       {onnx.__version__}')
print(f'onnxruntime: {onnxruntime.__version__}')
print(f'onnxscript:  {getattr(onnxscript, "__version__", "installed")}')
print('Зависимости готовы. Можно запускать ячейку 1 Setup.')


In [2]:
# @title 1. Mount Drive + basic imports
import os, sys, glob, math, warnings, io, json
from pathlib import Path

# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- Install missing packages ---
!pip install onnx onnxruntime -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
import onnx

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo 'No GPU'


## 2. Configuration

In [3]:
# @title 2. Paths, Constants & Fixed Hyperparameters

# --- Google Drive paths ---
DRIVE_BASE   = '/content/drive/MyDrive/RNN'
DATA_DIR     = os.path.join(DRIVE_BASE, 'data')
OUTPUT_DIR   = os.path.join(DRIVE_BASE, 'output')

# --- Trading parameters (MATCH PrepareData.mq5!) ---
SL_PIPS      = 15.0
TP_PIPS      = 20.0
HOLD_BARS    = 20
INITIAL_CAPITAL = 700.0

# --- Training constants ---
N_FEATURES   = 16       # признаков на бар (совпадает с MQL)
N_CLASSES    = 2        # DEAL / UNDEAL
BATCH_SIZE   = 64
FINAL_EPOCHS = 70       # эпох для финального обучения

# --- Trades baseline: N months in test x 3 trades/month ---
MIN_TRADES_PER_MONTH = 3

# --- Fixed hyperparameters for direct training ---
BUY_CONFIG = {
    'seq_len':        7,
    'norm_window':    150,
    'hidden_size':    112,
    'num_layers':     3,
    'dropout':        0.30,
    'lr':             0.004970,
    'conf_threshold': 0.5,
}

SELL_CONFIG = {
    'seq_len':        7,
    'norm_window':    200,
    'hidden_size':    80,
    'num_layers':     2,
    'dropout':        0.15,
    'lr':             0.004631,
    'conf_threshold': 0.8,
}

MODELS_CONFIG = [
    ('BUY',  BUY_CONFIG),
    ('SELL', SELL_CONFIG),
]

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Data:   {DATA_DIR}')
print(f'Output: {OUTPUT_DIR}')
print(f'\nBUY config:    {BUY_CONFIG}')
print(f'SELL config:   {SELL_CONFIG}')


## 3. Data Loading

In [11]:
# @title 3. Adaptive normalization + CSV loader

def adaptive_normalize(X, eps=1e-6):
    """Adaptive normalization (ddof=0, matches NN_ApplyRollingNorm in MQL)."""
    mean = X.mean(axis=1, keepdims=True)
    std  = X.std(axis=1, ddof=0, keepdims=True)
    std  = np.maximum(std, eps)
    return (X - mean) / std


def load_and_split_csv(filepath, seq_len, norm_window, eps=1e-6):
    """
    Load CSV, adaptive normalize, split 80/20 chronologically.
    Returns: train_loader, test_loader, df_test, seq_len, n_features, num_classes, n_months
    """
    df = pd.read_csv(filepath)
    has_datetime = 'datetime' in df.columns
    if has_datetime:
        df['datetime'] = pd.to_datetime(df['datetime'], format='%Y.%m.%d %H:%M')
        df = df.sort_values('datetime').reset_index(drop=True)

    feature_cols = [c for c in df.columns if c.startswith('lag')]
    if 'label' not in df.columns:
        raise KeyError(f"Column 'label' not found in {filepath}")

    n_bars = len(feature_cols) // N_FEATURES
    if n_bars * N_FEATURES != len(feature_cols):
        raise ValueError(f"Feature cols ({len(feature_cols)}) not divisible by N_FEATURES ({N_FEATURES})")
    if seq_len > n_bars:
        raise ValueError(f"seq_len ({seq_len}) > bars per row ({n_bars}). Reduce seq_len.")

    # Rolling windows for normalization
    if norm_window % n_bars != 0:
        raise ValueError(f"norm_window ({norm_window}) must be divisible by n_bars ({n_bars})")
    rows_per_window = norm_window // n_bars
    skip = rows_per_window
    if len(df) <= skip:
        raise ValueError(f"Need > {skip} rows, got {len(df)}")

    x_raw = df[feature_cols].values.astype(np.float64)
    x_raw = x_raw.reshape(-1, n_bars, N_FEATURES)

    chunks = np.lib.stride_tricks.sliding_window_view(x_raw, rows_per_window, axis=0)
    chunks = chunks[:, ::-1]
    X_raw = chunks.reshape(chunks.shape[0], -1, N_FEATURES)

    X_raw = X_raw[skip - rows_per_window + 1:]
    y = df['label'].values[skip:].astype(int)
    results = df['result'].values[skip:].astype(np.float64) if 'result' in df.columns else np.zeros(len(y))

    # Adaptive normalize
    X = adaptive_normalize(X_raw, eps)
    X = X[:, :seq_len, :]

    # Split chronologically 80/20
    n = X.shape[0]
    n_train = int(n * 0.8)

    X_train, X_test = X[:n_train], X[n_train:]
    y_train, y_test = y[:n_train], y[n_train:]
    results_test = results[n_train:]

    # Number of months in test set (for trades baseline)
    n_months = 1
    if has_datetime and n - n_train > 0:
        dt_test = df['datetime'].iloc[n_train + skip:]
        if len(dt_test) > 0:
            n_months = max(1, math.ceil((dt_test.iloc[-1] - dt_test.iloc[0]).days / 30.44))

    # One-hot
    num_classes = len(np.unique(y))
    y_train_oh = np.eye(num_classes)[y_train]
    y_test_oh  = np.eye(num_classes)[y_test]

    # DataFrame slice for backtest
    df_test = df.iloc[n_train + skip:].reset_index(drop=True).copy()
    df_test['_result'] = results_test

    print(f"  {os.path.basename(filepath)}: {len(df)} rows, train={n_train}, test={n - n_train}")
    print(f"    n_bars={n_bars}, norm_window={norm_window}, seq_len={seq_len}, test_months={n_months}")
    print(f"    Classes: {dict(zip(*np.unique(y, return_counts=True)))}")

    # Datasets
    train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train_oh))
    test_ds  = TensorDataset(torch.FloatTensor(X_test),  torch.FloatTensor(y_test_oh))
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, test_loader, df_test, seq_len, N_FEATURES, num_classes, n_months


# --- Find CSV files (search in RNN/data AND RNN itself) ---
def find_csvs(pattern):
    found = []
    for d in (DATA_DIR, DRIVE_BASE):
        if os.path.isdir(d):
            found += sorted(glob.glob(os.path.join(d, pattern)))
    # dedupe preserving order
    seen = set()
    out = []
    for f in found:
        if f not in seen:
            seen.add(f)
            out.append(f)
    return out

buy_csvs  = find_csvs('data_BUY_*.csv')
sell_csvs = find_csvs('data_SELL_*.csv')

print(f'Search dirs:\n  {DATA_DIR}\n  {DRIVE_BASE}')
print(f'BUY:  {[os.path.basename(f) for f in buy_csvs]}')
print(f'SELL: {[os.path.basename(f) for f in sell_csvs]}')

if not buy_csvs or not sell_csvs:
    print(f'\nERROR: data_BUY_*.csv / data_SELL_*.csv not found.\n')
    print('What is in the folders:')
    for d in (DRIVE_BASE, DATA_DIR):
        if os.path.isdir(d):
            print(f'  {d}:')
            for f in sorted(os.listdir(d)):
                print(f'    {f}')
    raise FileNotFoundError('CSV missing')


## 4. GRU Model

In [5]:
# @title 4. GRU Classifier

class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.gru = nn.GRU(input_size, hidden_size, num_layers,
                          batch_first=True,
                          dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        out, _ = self.gru(x, h0)
        out = out[:, -1, :]
        out = self.dropout(out)
        return self.fc(out)


## 5. Backtest Engine

In [6]:
# @title 5. Backtest: simulate trades, compute 4 metrics + score

def backtest(model, test_loader, df_test, threshold, n_months, device, initial_capital=700.0):
    """
    Simulate trading chronologically on test set.
    Returns dict: win_rate, max_dd, profit_factor, z_score, n_trades, score, equity, trades.
    Score = PF * WR * (1-DD) * sqrt(trades / baseline)
    baseline = n_months * MIN_TRADES_PER_MONTH
    """
    model.eval()
    all_probs = []
    with torch.no_grad():
        for X_batch, _ in test_loader:
            y_pred = model(X_batch.to(device))
            probs = torch.softmax(y_pred, dim=1)[:, 1]  # class DEAL / profitable
            all_probs.extend(probs.cpu().numpy())

    all_probs = np.array(all_probs)

    # Simulate trades
    trades = []
    for i, prob in enumerate(all_probs):
        if prob > threshold and i < len(df_test):
            r = df_test.iloc[i]['_result']
            trades.append(float(r))

    n_trades = len(trades)
    if n_trades == 0:
        return {'win_rate': 0.0, 'max_dd': 1.0, 'profit_factor': 0.0,
                'z_score': 0.0, 'n_trades': 0, 'total_return': 0.0,
                'avg_trade': 0.0, 'score': -999.0, 'equity': [], 'trades': []}

    trades = np.array(trades)
    n_wins = int(np.sum(trades > 0))
    win_rate = n_wins / n_trades

    # Equity curve
    equity = initial_capital + np.cumsum(trades)
    equity_full = np.insert(equity, 0, initial_capital)
    peak = np.maximum.accumulate(equity_full)
    dd = (peak - equity_full) / np.maximum(peak, 1e-9)
    max_dd = float(np.max(dd))

    # Profit Factor
    gross_profit  = np.sum(trades[trades > 0])
    gross_loss    = abs(np.sum(trades[trades < 0]))
    profit_factor = float(gross_profit / gross_loss) if gross_loss > 0 else (float('inf') if gross_profit > 0 else 0.0)

    total_return = float(np.sum(trades))
    avg_trade    = total_return / n_trades

    # Z-Score: statistical significance
    if n_trades > 1 and 0 < win_rate < 1:
        z_score = float((win_rate - 0.5) * math.sqrt(n_trades) / math.sqrt(win_rate * (1 - win_rate)))
    else:
        z_score = 0.0

    # SCORE: PF * WR * (1-DD) * sqrt(trades / baseline)
    baseline = max(1, n_months * MIN_TRADES_PER_MONTH)
    pf_clip = min(profit_factor, 10.0)  # cap extreme PF
    score = pf_clip * win_rate * (1.0 - max_dd) * math.sqrt(n_trades / baseline)

    return {
        'win_rate':      win_rate,
        'max_dd':        max_dd,
        'profit_factor': profit_factor,
        'z_score':       z_score,
        'n_trades':      n_trades,
        'total_return':  total_return,
        'avg_trade':     avg_trade,
        'score':         score,
        'equity':        equity.tolist(),
        'trades':        trades.tolist(),
    }


def plot_backtest(metrics, title, save_path=None):
    if not metrics.get('equity'):
        print('  No trades to plot.')
        return
    equity = np.array(metrics['equity'])
    equity_full = np.insert(equity, 0, INITIAL_CAPITAL)
    peak = np.maximum.accumulate(equity_full)
    dd = (peak - equity_full) / np.maximum(peak, 1e-9)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    ax1.plot(equity_full, 'b-', linewidth=1.0, label='Equity')
    ax1.plot(peak, 'g--', linewidth=0.8, alpha=0.7, label='Peak')
    ax1.fill_between(range(len(equity_full)), equity_full, peak, alpha=0.15, color='red')
    ax1.axhline(y=INITIAL_CAPITAL, color='gray', linestyle=':', alpha=0.5)
    ax1.set_ylabel('Equity')
    ax1.set_title(title)
    ax1.legend(loc='upper left')
    ax1.grid(True, alpha=0.3)
    ax2.fill_between(range(len(dd)), dd * 100, 0, alpha=0.3, color='red')
    ax2.plot(dd * 100, 'r-', linewidth=0.8)
    ax2.set_ylabel('Drawdown %')
    ax2.set_xlabel('Trade #')
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'  Chart: {save_path}')
    plt.show()


## 6. Training

In [7]:
# @title 6. Train one model

def train_one_model(model, train_loader, epochs, lr, device, verbose=True):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.1)
    best_acc = 0.0
    best_state = None

    it = tqdm(range(epochs), desc='Train', leave=False) if verbose else range(epochs)
    for epoch in it:
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch.argmax(dim=1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * X_batch.size(0)
            correct   += (y_pred.argmax(dim=1) == y_batch.argmax(dim=1)).sum().item()
            total     += X_batch.size(0)
        train_loss = total_loss / total
        train_acc  = correct / total
        scheduler.step()
        if train_acc > best_acc:
            best_acc = train_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if verbose and hasattr(it, 'set_postfix'):
            it.set_postfix({'loss': f'{train_loss:.4f}', 'acc': f'{train_acc:.4f}'})
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_acc, train_loss


## 7. Final Training + Backtest

In [8]:
# @title 7. Train with fixed params + backtest for each mode

def final_train_and_evaluate(csv_path, params, mode_name):
    seq_len      = params['seq_len']
    norm_window  = params['norm_window']
    hidden_size  = params['hidden_size']
    num_layers   = params['num_layers']
    dropout      = params['dropout']
    lr           = params['lr']
    conf_thresh  = params['conf_threshold']

    print(f"\n{'='*70}")
    print(f"FINAL: {mode_name} | s={seq_len} nw={norm_window} h={hidden_size} L={num_layers} dr={dropout:.2f} "
          f"lr={lr:.6f} th={conf_thresh:.2f}")
    print(f"{'='*70}")

    train_loader, test_loader, df_test, seq_len_actual, n_features, num_classes, n_months = \
        load_and_split_csv(csv_path, seq_len=seq_len, norm_window=norm_window)

    model = GRUClassifier(n_features, hidden_size, num_layers, num_classes, dropout)
    model, train_acc, train_loss = train_one_model(model, train_loader, FINAL_EPOCHS, lr, DEVICE, verbose=True)
    print(f"\n  Train Acc: {train_acc:.4f}, Loss: {train_loss:.4f}")

    metrics = backtest(model, test_loader, df_test, conf_thresh, n_months, DEVICE, INITIAL_CAPITAL)
    baseline = max(1, n_months * MIN_TRADES_PER_MONTH)

    print(f"\n  BACKTEST {mode_name} (test={n_months}mo, baseline={baseline} trades):")
    print(f"  {'─'*50}")
    print(f"  Trades:        {metrics['n_trades']}")
    print(f"  Win Rate:      {metrics['win_rate']:.4f} ({metrics['win_rate']*100:.1f}%)")
    print(f"  Max Drawdown:  {metrics['max_dd']:.4f} ({metrics['max_dd']*100:.1f}%)")
    print(f"  Profit Factor: {metrics['profit_factor']:.2f}")
    print(f"  Z-Score:       {metrics['z_score']:.2f}" + (' *' if abs(metrics['z_score']) > 2.0 else ''))
    print(f"  Total Return:  {metrics['total_return']:.2f}")
    print(f"  Avg Trade:     {metrics['avg_trade']:.4f}")
    print(f"  Score:         {metrics['score']:.4f}")
    print(f"  {'─'*50}")

    return model, metrics, seq_len, n_features, norm_window


# --- Loop over BUY/SELL configs ---
results = {}
for mode_name, config in MODELS_CONFIG:
    csv_path = buy_csvs[0] if mode_name == 'BUY' else sell_csvs[0]
    model, metrics, seq_len, n_features, norm_window = final_train_and_evaluate(csv_path, config, mode_name)
    results[mode_name] = {
        'model': model, 'metrics': metrics,
        'seq_len': seq_len, 'n_features': n_features,
        'norm_window': norm_window,
    }

buy_model   = results['BUY']['model']
buy_metrics = results['BUY']['metrics']
buy_seq_len = results['BUY']['seq_len']
buy_nf      = results['BUY']['n_features']
buy_nw      = results['BUY']['norm_window']

sell_model  = results['SELL']['model']
sell_metrics = results['SELL']['metrics']
sell_seq_len = results['SELL']['seq_len']
sell_nf      = results['SELL']['n_features']
sell_nw      = results['SELL']['norm_window']

# --- Plot ---
plot_backtest(buy_metrics,  f'BUY  | WR={buy_metrics["win_rate"]:.3f}  PF={buy_metrics["profit_factor"]:.2f}  Z={buy_metrics["z_score"]:.2f}',
              os.path.join(OUTPUT_DIR, 'buy_backtest.png'))
plot_backtest(sell_metrics, f'SELL | WR={sell_metrics["win_rate"]:.3f}  PF={sell_metrics["profit_factor"]:.2f}  Z={sell_metrics["z_score"]:.2f}',
              os.path.join(OUTPUT_DIR, 'sell_backtest.png'))


## 8. ONNX Export + RNN_Scaler.mqh

In [9]:
# @title 8. Export ONNX + generate MQL config

def export_to_onnx(model, seq_len, n_features, out_path, name):
    """Export to self-contained ONNX (no .data files). Required for OnnxCreateFromBuffer."""
    model.eval()
    model.to('cpu')
    dummy = torch.randn(1, seq_len, n_features)
    tmp = out_path + '.tmp'
    old = sys.stdout
    sys.stdout = io.StringIO()
    try:
        torch.onnx.export(model, dummy, tmp,
                          input_names=['input'], output_names=['output'],
                          dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
                          opset_version=18)
    finally:
        sys.stdout = old
    m = onnx.load(tmp)
    raw_bytes = m.SerializeToString()
    if b'.onnx.data' in raw_bytes:
        raise RuntimeError(f'{name}: external .data reference!')
    with open(out_path, 'wb') as f:
        f.write(raw_bytes)
    for x in (tmp, tmp + '.data', out_path + '.data'):
        if os.path.exists(x): os.remove(x)
    kb = os.path.getsize(out_path) / 1024
    print(f'  {name}: {out_path} ({kb:.1f} KB)')


def generate_scaler_mqh(buy_seq_len, buy_norm_window, sell_seq_len, sell_norm_window, n_features, eps, out_path):
    lines = [
        '//+------------------------------------------------------------------+',
        '//|                                              RNN_Scaler.mqh      |',
        '//|   Адаптивная нормализация.                                     |',
        '//|   Сгенерировано: Colab.                                        |',
        '//+------------------------------------------------------------------+',
        '#ifndef RNN_SCALER_MQH',
        '#define RNN_SCALER_MQH',
        '',
        f'#define NN_BUY_NORM_WINDOW  {buy_norm_window}   // BUY normalization window',
        f'#define NN_BUY_SEQ_LEN      {buy_seq_len}       // BUY input window',
        f'#define NN_SELL_NORM_WINDOW {sell_norm_window}   // SELL normalization window',
        f'#define NN_SELL_SEQ_LEN     {sell_seq_len}       // SELL input window',
        f'#define NN_MAX_NORM_WINDOW  {max(buy_norm_window, sell_norm_window)}',
        f'#define NN_FEATURES      {n_features}    // признаков на бар',
        f'#define NN_NORM_STD_EPS  {eps:g}         // epsilon',
        '',
        '/* Direction-neutral aliases for existing data preparation tools. */',
        '#define NN_NORM_WINDOW NN_BUY_NORM_WINDOW',
        '#define NN_SEQ_LEN     NN_BUY_SEQ_LEN',
        '',
        '#endif // RNN_SCALER_MQH',
        '',
    ]
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f'  RNN_Scaler.mqh -> {out_path}')


# --- Both models share the feature count; their windows are independent. ---
if buy_nf != sell_nf:
    raise ValueError(f'BUY/SELL feature count differs: {buy_nf} vs {sell_nf}')
generate_scaler_mqh(buy_seq_len, buy_nw, sell_seq_len, sell_nw, buy_nf, 1e-6,
                     os.path.join(OUTPUT_DIR, 'RNN_Scaler.mqh'))

# --- Export ONNX ---
onnx_dir = os.path.join(OUTPUT_DIR, 'onnx')
os.makedirs(onnx_dir, exist_ok=True)
export_to_onnx(buy_model,  buy_seq_len,  buy_nf, os.path.join(onnx_dir, 'buy_model.onnx'),  'buy')
export_to_onnx(sell_model, sell_seq_len, sell_nf, os.path.join(onnx_dir, 'sell_model.onnx'), 'sell')


## 9. Save All Artifacts

In [10]:
# @title 9. Save models + summary to Drive

# --- Save .pth ---
models_dir = os.path.join(OUTPUT_DIR, 'models')
os.makedirs(models_dir, exist_ok=True)
torch.save(buy_model.state_dict(),  os.path.join(models_dir, 'model_buy_final.pth'))
torch.save(sell_model.state_dict(), os.path.join(models_dir, 'model_sell_final.pth'))
print(f'Models: {models_dir}/')

# --- Summary JSON ---
def safe(v):
    if isinstance(v, (np.floating,)): return float(v)
    if isinstance(v, (np.integer,)):  return int(v)
    if isinstance(v, np.ndarray):     return v.tolist()
    return v

summary = {
    'buy_metrics':  {k: safe(v) for k, v in buy_metrics.items() if k not in ('equity','trades')},
    'sell_metrics': {k: safe(v) for k, v in sell_metrics.items() if k not in ('equity','trades')},
    'buy_params':   {k: safe(v) for k, v in BUY_CONFIG.items()},
    'sell_params':  {k: safe(v) for k, v in SELL_CONFIG.items()},
    'sl_pips':  SL_PIPS,
    'tp_pips':  TP_PIPS,
}
with open(os.path.join(OUTPUT_DIR, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=safe)
print(f'Summary: {OUTPUT_DIR}/summary.json')

# --- Final report ---
print(f"\n{'='*70}")
print(f"DONE! Files in: {OUTPUT_DIR}/")
print(f"{'='*70}")
print(f"\nDownload to local MT5:\n")
print(f"  1. {onnx_dir}/buy_model.onnx  -> Common\\Files\\RNN\\buy_model.onnx")
print(f"  2. {onnx_dir}/sell_model.onnx -> Common\\Files\\RNN\\sell_model.onnx")
print(f"  3. {OUTPUT_DIR}/RNN_Scaler.mqh -> MQL5\\Experts\\RNN\\RNN_Scaler.mqh")
print(f"\nBUY:  WR={buy_metrics['win_rate']:.3f} DD={buy_metrics['max_dd']:.3f} PF={buy_metrics['profit_factor']:.2f} Z={buy_metrics['z_score']:.2f} N={buy_metrics['n_trades']}")
print(f"SELL: WR={sell_metrics['win_rate']:.3f} DD={sell_metrics['max_dd']:.3f} PF={sell_metrics['profit_factor']:.2f} Z={sell_metrics['z_score']:.2f} N={sell_metrics['n_trades']}")
